# ML-06 — Signal Audit

March 2026 mid-panel audit; June remains sealed.


## 1. Distributions

I audit impressions and CTR versus position over a 21-day decision snapshot.


In [1]:
from pathlib import Path
import os,duckdb,pandas as pd,numpy as np
p=list((Path.home()/'.cache'/'huggingface'/'hub').rglob('fact_content_daily_performance/month=2026-03/data_0.parquet'))

if p: path=str(p[0])
else:
    try:
        from google.colab import userdata; 
        token=userdata.get('HF_TOKEN')
    except Exception: 
        token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if not token: 
        raise RuntimeError('Set HF_TOKEN as a Colab Secret or environment variable.')
    from huggingface_hub import hf_hub_download; 
    path=hf_hub_download('FlyRank/internship-warehouse','fact_content_daily_performance/month=2026-03/data_0.parquet',repo_type='dataset',token=token)

safe=path.replace(chr(39),chr(39)*2); 
con=duckdb.connect(); 
con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{safe}')")

d=con.sql("SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) impressions_21d,SUM(gsc_clicks) clicks_21d,SUM(gsc_sum_position)/NULLIF(SUM(gsc_impressions),0) avg_position_21d FROM march WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21' AND gsc_data_available IS TRUE GROUP BY 1,2 HAVING SUM(gsc_impressions)>0").df(); 
d['ctr_pct']=100*d.clicks_21d/d.impressions_21d; 
print('Items:',len(d)); 
display(d[['impressions_21d','clicks_21d','avg_position_21d','ctr_pct']].describe().round(3))


Items: 162301


,impressions_21d,clicks_21d,avg_position_21d,ctr_pct
count,162301.000,162301.000,162301.000,162301.000
mean,1103.956,3.348,15.972,0.467
std,3559.245,17.971,18.012,3.794
min,1.000,0.000,0.000,0.000
25%,18.000,0.000,4.888,0.000
50%,138.000,0.000,8.218,0.000
75%,768.000,2.000,20.333,0.220
max,273012.000,3398.000,309.000,100.000


## 2. Signal tests

Both tables show n and use only pre-decision data.


In [2]:
d['position_bucket']=pd.cut(d.avg_position_21d,[0,3,5,10,20,1e9],labels=['1-3','4-5','6-10','11-20','21+']); 
pos=d.groupby('position_bucket',observed=True).agg(n=('content_hash_id','size'),impressions=('impressions_21d','sum'),clicks=('clicks_21d','sum')).assign(weighted_ctr_pct=lambda x:100*x.clicks/x.impressions).reset_index(); 
print('Signal 1 CTR vs position — CONFIRMED (flag-linked): CTR declines as position worsens.'); 
display(pos.round(3))

d['volume_bucket']=pd.cut(d.impressions_21d,[0,99,499,1999,9999,float('inf')],labels=['1-99','100-499','500-1,999','2,000-9,999','10,000+']); 
vol=d.groupby('volume_bucket',observed=True).agg(n=('content_hash_id','size'),median_clicks=('clicks_21d','median'),total_clicks=('clicks_21d','sum')).reset_index(); 
print('Signal 2 volume — CONFIRMED: larger buckets contain more observed clicks; priority is not causal lift.'); 
display(vol.round(3))


Signal 1 CTR vs position — CONFIRMED (flag-linked): CTR declines as position worsens.


,position_bucket,n,impressions,clicks,weighted_ctr_pct
0,1-3,16821,27380357.0,116203.0,0.424
1,4-5,24997,47140183.0,172840.0,0.367
2,6-10,50660,44903175.0,135129.0,0.301
3,11-20,27411,19410193.0,62790.0,0.323
4,21+,41159,40336775.0,56310.0,0.140


Signal 2 volume — CONFIRMED: larger buckets contain more observed clicks; priority is not causal lift.


,volume_bucket,n,median_clicks,total_clicks
0,1-99,73389,0.0,6204.0
1,100-499,38460,0.0,24939.0
2,"500-1,999",29692,2.0,97304.0
3,"2,000-9,999",17573,8.0,229629.0
4,"10,000+",3187,32.0,185249.0


## 3. Flag-linked test

Low CTR is most actionable when visible at positions 4–20.


In [3]:
d['rule_eligible']=(d.impressions_21d>=500)&d.avg_position_21d.between(4,20)&(d.ctr_pct<.5); 
summary=d.assign(bucket=np.where(d.rule_eligible,'eligible','not_eligible')).groupby('bucket').agg(n=('content_hash_id','size'),median_impressions=('impressions_21d','median'),median_ctr_pct=('ctr_pct','median'),median_position=('avg_position_21d','median')).reset_index(); 
display(summary.round(3)); 
print('CONFIRMED: eligible items are measurable, visible, low-CTR opportunities.')


,bucket,n,median_impressions,median_ctr_pct,median_position
0,eligible,23190,1437.0,0.156,7.235
1,not_eligible,139111,84.0,0.000,8.725


CONFIRMED: eligible items are measurable, visible, low-CTR opportunities.


## 4. Practice and limitation

Review visible low-CTR pages first; volume indicates possible reach, not guaranteed lift. SERP features, brand intent, and experiments are missing, so this is decision support, not a causal claim.


In [4]:
assert len(pos)>0 and len(vol)>0; 
print('Self-check passed; position-null rows are excluded only from position buckets.')


Self-check passed; position-null rows are excluded only from position buckets.


## Self-check

- Two bucket tables with n and verdicts
- Flag-linked CTR/position test
- No future, label-derived, or token inputs
